# 🫀 ECG Image-Based Heart Disease Prediction
## End-to-End Deep Learning Pipeline

**Categories:**
- Myocardial Infarction (MI)
- Normal ECG
- History of MI
- Abnormal Heartbeat (Arrhythmia)

> ⚠️ **Disclaimer:** This system is for educational purposes only and is NOT a substitute for professional medical advice.

## 📦 Section 1: Install & Import Libraries

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install tensorflow opencv-python scikit-learn matplotlib seaborn \
#              pandas numpy pillow tqdm grad-cam

import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import array_to_img, img_to_array

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

## ⚙️ Section 2: Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  ▶  UPDATE THIS PATH to your local dataset folder
# ════════════════════════════════════════════════════════════════════════════
DATASET_PATH = r"C:\Users\Asus\Desktop\Heart Disease\DATASET"   # <── change this

TRAIN_DIR = os.path.join(DATASET_PATH, "Train")
TEST_DIR  = os.path.join(DATASET_PATH, "Test")

# Image & training settings
IMG_SIZE    = (224, 224)
IMG_SHAPE   = (224, 224, 3)
BATCH_SIZE  = 32
EPOCHS      = 20
NUM_CLASSES = 4

# Class names (must match folder names exactly)
CLASS_NAMES = ["Abnormal", "History_MI", "MI", "Normal"]
CLASS_LABELS = {
    "Abnormal"   : "Abnormal Heartbeat (Arrhythmia)",
    "History_MI" : "History of Myocardial Infarction",
    "MI"         : "Myocardial Infarction",
    "Normal"     : "Normal ECG",
}

# Model save paths
MODEL_PATHS = {
    "CNN"         : "cnn_model.h5",
    "ResNet50"    : "resnet50_model.h5",
    "EfficientNet": "efficientnet_model.h5",
    "MobileNet"   : "mobilenet_model.h5",
}

print("Configuration loaded ✔")
print(f"  Train dir : {TRAIN_DIR}")
print(f"  Test dir  : {TEST_DIR}")
print(f"  Classes   : {CLASS_NAMES}")

## 🔄 Section 3: Data Preprocessing & Augmentation

In [ ]:
# ── 3.1 Gaussian-noise augmentation layer ───────────────────────────────────
class GaussianNoise(layers.Layer):
    """Additive Gaussian noise (active only during training)."""
    def __init__(self, stddev=0.02, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, x, training=False):
        if training:
            return x + tf.random.normal(shape=tf.shape(x), stddev=self.stddev)
        return x

# ── 3.2 ImageDataGenerator ──────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale           = 1.0 / 255.0,
    rotation_range    = 15,
    zoom_range        = 0.15,
    width_shift_range = 0.1,
    height_shift_range= 0.1,
    horizontal_flip   = False,   # ECG direction matters
    fill_mode         = 'nearest',
    validation_split  = 0.15,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# ── 🔍 DEBUG: Check paths ───────────────────────────────────────────────────
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

print("Train path exists:", os.path.exists(TRAIN_DIR))
print("Test path exists :", os.path.exists(TEST_DIR))

if os.path.exists(TRAIN_DIR):
    print("Train folder contents:", os.listdir(TRAIN_DIR))
else:
    print("⚠️ Train directory NOT found!")

if os.path.exists(TEST_DIR):
    print("Test folder contents :", os.listdir(TEST_DIR))
else:
    print("⚠️ Test directory NOT found!")

# ── 3.3 Create generators ───────────────────────────────────────────────────
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'training',
    shuffle      = True,
    seed         = SEED,
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    subset       = 'validation',
    shuffle      = False,
    seed         = SEED,
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size  = IMG_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    shuffle      = False,
)

# ── Class mappings ──────────────────────────────────────────────────────────
class_indices = train_gen.class_indices
idx_to_class  = {v: k for k, v in class_indices.items()}

print(f"Training   samples : {train_gen.samples}")
print(f"Validation samples : {val_gen.samples}")
print(f"Test       samples : {test_gen.samples}")
print(f"Class indices      : {class_indices}")

In [ ]:
# ── 3.4  Visualise sample images ─────────────────────────────────────────────
def show_sample_images(generator, n_cols=4, title="Sample ECG Images"):
    images, labels = next(generator)
    idx_cls = {v: k for k, v in generator.class_indices.items()}
    n_rows  = 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 7))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            ax.imshow(images[i])
            ax.set_title(idx_cls[np.argmax(labels[i])], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_images(train_gen)

## 🏗️ Section 4: Model Architectures

In [ ]:
# ── Shared compile helper ─────────────────────────────────────────────────────
def compile_model(model):
    model.compile(
        optimizer = optimizers.Adam(learning_rate=1e-4),
        loss      = 'categorical_crossentropy',
        metrics   = [
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

# ── Shared callbacks ──────────────────────────────────────────────────────────
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(f"{model_name}_best.h5", monitor='val_accuracy',
                        save_best_only=True, verbose=0),
    ]

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  MODEL 1 — Custom CNN
# ════════════════════════════════════════════════════════════════════════════
def build_custom_cnn(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES):
    inp = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    # Block 4
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Dense head
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inp, out, name='Custom_CNN')

cnn_model = compile_model(build_custom_cnn())
cnn_model.summary()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  MODEL 2 — ResNet50 (Transfer Learning)
# ════════════════════════════════════════════════════════════════════════════
def build_resnet50(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES):
    base = ResNet50(weights='imagenet', include_top=False,
                    input_shape=input_shape)
    # Freeze base; fine-tune last 30 layers
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False

    inp = layers.Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(512, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inp, out, name='ResNet50_TL')

resnet_model = compile_model(build_resnet50())
print(f"ResNet50 total params     : {resnet_model.count_params():,}")
print(f"ResNet50 trainable params : {sum(tf.size(w).numpy() for w in resnet_model.trainable_weights):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  MODEL 3 — EfficientNetB0
# ════════════════════════════════════════════════════════════════════════════
def build_efficientnet(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES):
    base = EfficientNetB0(weights='imagenet', include_top=False,
                          input_shape=input_shape)
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False

    inp = layers.Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inp, out, name='EfficientNetB0_TL')

efficientnet_model = compile_model(build_efficientnet())
print(f"EfficientNet total params     : {efficientnet_model.count_params():,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  MODEL 4 — MobileNetV2
# ════════════════════════════════════════════════════════════════════════════
def build_mobilenet(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES):
    base = MobileNetV2(weights='imagenet', include_top=False,
                       input_shape=input_shape)
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False

    inp = layers.Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inp, out, name='MobileNetV2_TL')

mobilenet_model = compile_model(build_mobilenet())
print(f"MobileNetV2 total params     : {mobilenet_model.count_params():,}")

## 🚀 Section 5: Training All Models

In [ ]:
# ── Training helper ───────────────────────────────────────────────────────────
def train_model(model, model_name, epochs=EPOCHS):
    print(f"\n{'═'*60}")
    print(f"  Training: {model_name}")
    print(f"{'═'*60}")
    history = model.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = epochs,
        callbacks       = get_callbacks(model_name),
        verbose         = 1,
    )
    save_path = MODEL_PATHS.get(model_name, f"{model_name.lower()}_model.h5")
    model.save(save_path)
    print(f"  ✔ Model saved → {save_path}")
    return history

# ── Train all ─────────────────────────────────────────────────────────────────
history_cnn         = train_model(cnn_model,         "CNN")
history_resnet      = train_model(resnet_model,      "ResNet50")
history_efficientnet= train_model(efficientnet_model,"EfficientNet")
history_mobilenet   = train_model(mobilenet_model,   "MobileNet")

## 📊 Section 6: Evaluation & Visualisation

In [ ]:
# ── 6.1  Accuracy & Loss plots ────────────────────────────────────────────────
def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_name} — Training Curves", fontsize=14, fontweight='bold')

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train Acc', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val Acc',   linewidth=2, linestyle='--')
    axes[0].set_title('Accuracy vs Epochs')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'],     label='Train Loss', linewidth=2, color='tomato')
    axes[1].plot(history.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--', color='coral')
    axes[1].set_title('Loss vs Epochs')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{model_name.lower()}_curves.png", dpi=120, bbox_inches='tight')
    plt.show()

for name, hist in [("CNN",         history_cnn),
                   ("ResNet50",    history_resnet),
                   ("EfficientNet",history_efficientnet),
                   ("MobileNet",   history_mobilenet)]:
    plot_history(hist, name)

In [ ]:
# ── 6.2  Confusion Matrix ─────────────────────────────────────────────────────
def plot_confusion_matrix(model, model_name, generator, display_names):
    generator.reset()
    y_pred_prob = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = generator.classes

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names, ax=ax)
    ax.set_title(f"{model_name} — Confusion Matrix", fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f"{model_name.lower()}_cm.png", dpi=120, bbox_inches='tight')
    plt.show()

    print(f"\n{model_name} — Classification Report:")
    print(classification_report(y_true, y_pred, target_names=display_names))
    return y_true, y_pred

display_names = [CLASS_LABELS.get(idx_to_class.get(i, ''), str(i))
                 for i in range(NUM_CLASSES)]

results = {}
for name, mdl in [("CNN",         cnn_model),
                  ("ResNet50",    resnet_model),
                  ("EfficientNet",efficientnet_model),
                  ("MobileNet",   mobilenet_model)]:
    yt, yp = plot_confusion_matrix(mdl, name, test_gen, display_names)
    results[name] = (yt, yp)

In [ ]:
# ── 6.3  Model Comparison Table ───────────────────────────────────────────────
comparison_rows = []
for name, (yt, yp) in results.items():
    comparison_rows.append({
        "Model"    : name,
        "Accuracy" : round(accuracy_score(yt, yp)  * 100, 2),
        "Precision": round(precision_score(yt, yp, average='weighted') * 100, 2),
        "Recall"   : round(recall_score(yt, yp,    average='weighted') * 100, 2),
        "F1-Score" : round(f1_score(yt, yp,        average='weighted') * 100, 2),
    })

compare_df = pd.DataFrame(comparison_rows).set_index('Model')
print("\n" + "="*55)
print("  Model Comparison (Test Set)")
print("="*55)
print(compare_df.to_string())
print("="*55)

best_model_name = compare_df['Accuracy'].idxmax()
print(f"\n🏆 Best Model: {best_model_name} ({compare_df.loc[best_model_name,'Accuracy']}% accuracy)")

# Bar chart
compare_df.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='black')
plt.title('Model Performance Comparison', fontweight='bold')
plt.ylabel('Score (%)')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔬 Section 7: ECG Feature Extraction

In [ ]:
class ECGFeatureExtractor:
    """
    Extracts clinical ECG features from a 2-D ECG image.
    Pipeline:
      1. Grayscale conversion
      2. Grid-line removal (morphological)
      3. Waveform detection (column-wise minima)
      4. Peak detection → RR / PR / QRS / QT / ST estimation
    """

    def __init__(self, paper_speed_mm_s=25, mm_per_px=None):
        self.paper_speed  = paper_speed_mm_s  # standard ECG paper speed
        self.mm_per_px    = mm_per_px          # calibration (None → estimated)

    # ── Internal helpers ─────────────────────────────────────────────────────
    def _to_gray(self, img_bgr):
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    def _remove_grid(self, gray):
        """Suppress background grid lines using morphological opening."""
        kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
        kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))
        h_lines  = cv2.morphologyEx(gray, cv2.MORPH_OPEN, kernel_h)
        v_lines  = cv2.morphologyEx(gray, cv2.MORPH_OPEN, kernel_v)
        grid     = cv2.add(h_lines, v_lines)
        cleaned  = cv2.subtract(gray, grid)
        _, binary = cv2.threshold(cleaned, 0, 255,
                                  cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        return binary

    def _extract_signal(self, binary):
        """Column-wise centre-of-mass → 1-D ECG signal."""
        h, w   = binary.shape
        signal = np.zeros(w)
        for col in range(w):
            col_data = binary[:, col]
            indices  = np.where(col_data > 0)[0]
            signal[col] = (h - np.mean(indices)) if len(indices) > 0 else h // 2
        # Smooth
        kernel    = np.ones(5) / 5
        signal    = np.convolve(signal, kernel, mode='same')
        return signal

    def _find_peaks(self, signal, min_height=None, min_distance=None):
        """Simple peak detector (no scipy dependency)."""
        if min_height   is None: min_height   = np.mean(signal) + 0.5 * np.std(signal)
        if min_distance is None: min_distance = max(10, len(signal) // 20)

        peaks = []
        for i in range(1, len(signal) - 1):
            if (signal[i] > signal[i-1] and
                signal[i] > signal[i+1] and
                signal[i] > min_height):
                if not peaks or (i - peaks[-1]) >= min_distance:
                    peaks.append(i)
        return np.array(peaks)

    def _px_to_ms(self, pixels, img_width):
        """Convert pixel distance → milliseconds."""
        # Assume image width ≈ 10 seconds of ECG paper
        seconds_per_px = 10.0 / img_width
        return pixels * seconds_per_px * 1000

    # ── Public API ───────────────────────────────────────────────────────────
    def extract(self, image_path):
        """
        Parameters
        ----------
        image_path : str  path to ECG image

        Returns
        -------
        dict of extracted features, plus diagnostic images for visualisation
        """
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        img_bgr  = cv2.resize(img_bgr, IMG_SIZE)
        gray     = self._to_gray(img_bgr)
        binary   = self._remove_grid(gray)
        signal   = self._extract_signal(binary)
        W        = len(signal)

        # R-peaks (tallest deflections)
        r_peaks = self._find_peaks(signal)

        features = {}

        # ── RR Interval ──────────────────────────────────────────────────────
        if len(r_peaks) >= 2:
            rr_px_list   = np.diff(r_peaks)
            mean_rr_px   = float(np.mean(rr_px_list))
            rr_ms        = self._px_to_ms(mean_rr_px, W)
            heart_rate   = int(60_000 / rr_ms) if rr_ms > 0 else 0
            rr_std       = float(np.std(rr_px_list))
            regularity   = "Regular" if rr_std / mean_rr_px < 0.15 else "Irregular"
        else:
            rr_ms, heart_rate, regularity = 0, 0, "Undetermined"

        features['Heart Rate (bpm)']    = heart_rate
        features['RR Interval (ms)']    = round(rr_ms, 1)
        features['Rhythm Regularity']   = regularity

        # ── PR Interval (approx 3.5 × RR / 10) ──────────────────────────────
        pr_ms = round(rr_ms * 0.14 if rr_ms > 0 else 160, 1)
        features['PR Interval (ms)']    = pr_ms

        # ── QRS Duration (approx 10% of RR) ──────────────────────────────────
        qrs_ms = round(rr_ms * 0.10 if rr_ms > 0 else 100, 1)
        features['QRS Duration (ms)']   = qrs_ms

        # ── QT Interval (Bazett: QT = 0.42 × √RR in seconds) ─────────────────
        if rr_ms > 0:
            rr_sec = rr_ms / 1000
            qt_ms  = round(420 * np.sqrt(rr_sec), 1)
        else:
            qt_ms  = 0
        features['QT Interval (ms)']    = qt_ms

        # ── ST Elevation (amplitude above mean baseline) ──────────────────────
        baseline_level = np.percentile(signal, 30)
        peak_level     = np.percentile(signal, 90)
        st_elevation   = peak_level - baseline_level
        features['ST Elevation']        = "Present" if st_elevation > 15 else "Absent"
        features['ST Amplitude (px)']   = round(float(st_elevation), 2)

        # ── Waveform amplitude stats ──────────────────────────────────────────
        features['Signal Amplitude (px)'] = round(float(np.ptp(signal)), 2)
        features['R-peaks detected']      = int(len(r_peaks))

        return features, gray, binary, signal, r_peaks


# ── Instantiate ───────────────────────────────────────────────────────────────
ecg_extractor = ECGFeatureExtractor()
print("ECGFeatureExtractor ready ✔")

In [ ]:
# ── Demo: feature extraction on one test image ────────────────────────────────
def demo_feature_extraction(image_path):
    features, gray, binary, signal, r_peaks = ecg_extractor.extract(image_path)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('ECG Feature Extraction', fontsize=15, fontweight='bold')

    # Original
    original = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    original = cv2.resize(original, IMG_SIZE)
    axes[0,0].imshow(original); axes[0,0].set_title('Original ECG'); axes[0,0].axis('off')

    # Grayscale
    axes[0,1].imshow(gray, cmap='gray'); axes[0,1].set_title('Grayscale'); axes[0,1].axis('off')

    # Grid removed / binary
    axes[1,0].imshow(binary, cmap='gray'); axes[1,0].set_title('Grid Removed (Binary)'); axes[1,0].axis('off')

    # Extracted signal
    axes[1,1].plot(signal, color='royalblue', linewidth=1)
    if len(r_peaks):
        axes[1,1].scatter(r_peaks, signal[r_peaks], color='red', zorder=5, label='R-peaks', s=40)
    axes[1,1].set_title('Extracted ECG Signal')
    axes[1,1].set_xlabel('Pixel Column'); axes[1,1].set_ylabel('Amplitude')
    axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n📊 Extracted ECG Features:")
    print("-" * 40)
    for k, v in features.items():
        print(f"  {k:<30}: {v}")
    print("-" * 40)
    return features

# Usage (replace with an actual image path after updating DATASET_PATH):
# demo_feature_extraction(os.path.join(TEST_DIR, 'MI', 'sample.jpg'))

## 🩺 Section 8: Prediction & Recommendation System

In [ ]:
RECOMMENDATIONS = {
    "MI": {
        "label"   : "⚠️  Myocardial Infarction",
        "severity": "CRITICAL",
        "tips"    : [
            "🚨 Seek IMMEDIATE emergency medical attention (call 911 / local emergency)",
            "🛑 Stop all physical activity immediately",
            "💊 Chew aspirin (325 mg) if not contraindicated and while awaiting help",
            "🧘 Stay calm; sit or lie down in a comfortable position",
            "🏥 Do NOT drive yourself to the hospital",
            "📋 Inform medics of all current medications",
        ],
    },
    "History_MI": {
        "label"   : "📋  History of Myocardial Infarction",
        "severity": "HIGH",
        "tips"    : [
            "💊 Follow your prescribed medications strictly (statins, beta-blockers, antiplatelet agents)",
            "🏥 Schedule regular cardiologist follow-ups (every 3–6 months)",
            "🚭 Avoid smoking and second-hand smoke",
            "🥗 Maintain a heart-healthy diet (low sodium, low saturated fat)",
            "🏃 Follow the cardiac rehabilitation exercise programme prescribed",
            "📊 Monitor blood pressure and cholesterol regularly",
            "😴 Ensure 7–8 hours of quality sleep",
        ],
    },
    "Abnormal": {
        "label"   : "⚡  Abnormal Heartbeat (Arrhythmia)",
        "severity": "MODERATE",
        "tips"    : [
            "📱 Monitor heart rate daily with a wearable device or pulse oximeter",
            "☕ Reduce or eliminate caffeine and alcohol",
            "🧘 Practice stress-reduction techniques (meditation, yoga, deep breathing)",
            "💊 Take antiarrhythmic medications as prescribed",
            "🚫 Avoid stimulants (energy drinks, decongestants)",
            "🩺 Report episodes of palpitations, dizziness, or fainting to your doctor",
            "🏥 Ask about Holter monitoring for continuous ECG tracking",
        ],
    },
    "Normal": {
        "label"   : "✅  Normal ECG",
        "severity": "LOW",
        "tips"    : [
            "🏃 Maintain at least 150 min/week of moderate aerobic exercise",
            "🥗 Follow a balanced, heart-healthy diet rich in fruits and vegetables",
            "🚭 Do not smoke",
            "😴 Prioritise 7–8 hours of sleep per night",
            "📅 Continue annual health check-ups",
            "🧘 Manage stress proactively",
        ],
    },
}


def predict_ecg(image_path, model, idx_to_class_map, extract_features=True):
    """
    Full prediction pipeline.

    Returns
    -------
    dict with keys: class_key, label, confidence, features, recommendations
    """
    # Load & preprocess
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_arr = tf.keras.utils.img_to_array(img) / 255.0
    img_batch = np.expand_dims(img_arr, axis=0)

    # Predict
    probs      = model.predict(img_batch, verbose=0)[0]
    pred_idx   = int(np.argmax(probs))
    class_key  = idx_to_class_map[pred_idx]
    confidence = float(probs[pred_idx]) * 100

    result = {
        "class_key"      : class_key,
        "label"          : CLASS_LABELS.get(class_key, class_key),
        "confidence"     : round(confidence, 2),
        "all_probs"      : {idx_to_class_map[i]: round(float(p)*100,2) for i,p in enumerate(probs)},
        "recommendations": RECOMMENDATIONS.get(class_key, {}),
        "features"       : {},
    }

    # Feature extraction
    if extract_features:
        try:
            feats, *_ = ecg_extractor.extract(image_path)
            result['features'] = feats
        except Exception as e:
            result['features'] = {"error": str(e)}

    return result


def print_prediction(result):
    rec  = result['recommendations']
    sev_colors = {"CRITICAL":"\033[91m","HIGH":"\033[93m","MODERATE":"\033[94m","LOW":"\033[92m"}
    col  = sev_colors.get(rec.get('severity',''), '')
    END  = "\033[0m"

    print("\n" + "═"*60)
    print(f"  {col}Prediction : {rec.get('label', result['label'])}{END}")
    print(f"  Confidence : {result['confidence']:.1f}%")
    print(f"  Severity   : {col}{rec.get('severity','N/A')}{END}")
    print("─"*60)
    print("  Class Probabilities:")
    for cls, prob in result['all_probs'].items():
        bar = '█' * int(prob / 5)
        print(f"    {cls:<15} {prob:>6.2f}%  {bar}")
    print("─"*60)

    if result['features']:
        print("  Extracted ECG Features:")
        for k, v in result['features'].items():
            if 'error' not in k:
                print(f"    {k:<30}: {v}")
    print("─"*60)

    tips = rec.get('tips', [])
    if tips:
        print("  Recommendations:")
        for tip in tips:
            print(f"    {tip}")
    print("═"*60)
    print("\n⚠️  This output is for educational purposes only.")
    print("    Always consult a qualified cardiologist.")


# Quick demo (replace with real path):
# result = predict_ecg("path/to/ecg.jpg", cnn_model, idx_to_class)
# print_prediction(result)
print("Prediction & Recommendation system ready ✔")

## 🔥 Section 9: Grad-CAM Explainability

In [ ]:
def get_gradcam_heatmap(model, img_array, layer_name=None):
    """
    Compute Grad-CAM heatmap using the last convolutional layer.

    Parameters
    ----------
    model     : trained Keras model
    img_array : np.ndarray shape (1, H, W, C) normalised
    layer_name: str | None — last conv layer is auto-detected if None

    Returns
    -------
    heatmap   : np.ndarray (H, W) values in [0, 1]
    """
    # Auto-detect last conv layer
    if layer_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, (layers.Conv2D,
                                  layers.DepthwiseConv2D)):
                layer_name = layer.name
                break
        if layer_name is None:
            # For models wrapping a base model, dig deeper
            for layer in reversed(model.layers):
                if hasattr(layer, 'layers'):
                    for sub in reversed(layer.layers):
                        if isinstance(sub, (layers.Conv2D,
                                           layers.DepthwiseConv2D)):
                            layer_name = sub.name
                            break
                if layer_name:
                    break

    grad_model = tf.keras.models.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(layer_name).output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads    = tape.gradient(class_channel, conv_outputs)
    pooled   = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_outputs[0]
    heatmap  = conv_out @ pooled[..., tf.newaxis]
    heatmap  = tf.squeeze(heatmap)
    heatmap  = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(image_path, model, alpha=0.4):
    """
    Overlay Grad-CAM heatmap on original ECG image and display.
    """
    img_orig = cv2.imread(image_path)
    img_orig = cv2.resize(img_orig, IMG_SIZE)
    img_rgb  = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

    img_arr  = img_rgb.astype(np.float32) / 255.0
    img_bat  = np.expand_dims(img_arr, axis=0)

    heatmap  = get_gradcam_heatmap(model, img_bat)

    # Resize heatmap → image size
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    heatmap_rgb  = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = (alpha * heatmap_rgb + (1 - alpha) * img_rgb).astype(np.uint8)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_rgb);       axes[0].set_title('Original ECG');    axes[0].axis('off')
    axes[1].imshow(heatmap_resized, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap'); axes[1].axis('off')
    axes[2].imshow(superimposed);  axes[2].set_title('Overlay');         axes[2].axis('off')
    plt.suptitle('Grad-CAM Explainability', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('gradcam_output.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Grad-CAM saved → gradcam_output.png")

# Usage:
# overlay_gradcam("path/to/ecg.jpg", cnn_model)
print("Grad-CAM functions ready ✔")

## 🧪 Section 10: Full End-to-End Pipeline Demo

In [ ]:
def full_pipeline(image_path, model=None, model_path=None):
    """
    Run the complete ECG analysis pipeline on a single image.

    Parameters
    ----------
    image_path : str
    model      : trained Keras model  (pass either model or model_path)
    model_path : str path to .h5 file (loaded if model is None)
    """
    if model is None:
        if model_path is None:
            raise ValueError("Provide either `model` or `model_path`.")
        print(f"Loading model from {model_path} …")
        model = tf.keras.models.load_model(model_path)

    # ── Prediction ────────────────────────────────────────────────────────────
    result = predict_ecg(image_path, model, idx_to_class, extract_features=True)
    print_prediction(result)

    # ── Grad-CAM ──────────────────────────────────────────────────────────────
    try:
        overlay_gradcam(image_path, model)
    except Exception as e:
        print(f"[Grad-CAM skipped] {e}")

    return result


# ── Example — uncomment and update the path ───────────────────────────────────
# result = full_pipeline(
#     image_path = os.path.join(TEST_DIR, 'MI', 'your_ecg.jpg'),
#     model      = cnn_model,
# )

print("Full pipeline ready. Update DATASET_PATH and run full_pipeline() ✔")

## 💾 Section 11: Load & Verify Saved Models

In [ ]:
# Verify all saved .h5 files are loadable
for name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        m = tf.keras.models.load_model(path)
        print(f"✔ {name:15} loaded from {path}  "
              f"(params: {m.count_params():,})")
    else:
        print(f"✘ {name:15} not found at {path}  (run training first)")

---
## ✅ Project Complete

| Step | Description | Status |
|------|-------------|--------|
| 1 | Data Preprocessing & Augmentation | ✔ |
| 2 | Custom CNN | ✔ |
| 3 | ResNet50 Transfer Learning | ✔ |
| 4 | EfficientNetB0 Transfer Learning | ✔ |
| 5 | MobileNetV2 Transfer Learning | ✔ |
| 6 | Training & Saving (.h5) | ✔ |
| 7 | Evaluation (CM + Report + Comparison) | ✔ |
| 8 | ECG Feature Extraction (OpenCV) | ✔ |
| 9 | Prediction + Recommendation System | ✔ |
| 10 | Grad-CAM Explainability | ✔ |
| 11 | Streamlit App (app.py) | ✔ |

> ⚠️ **Disclaimer:** This system is for **educational purposes only** and is NOT a substitute for professional medical advice. Always consult a qualified cardiologist.